The same lab as a web page, with the folded answers: https://ml.wxing.me/materials/lesson9/lab.html

# Lab 9: Train a Neural Network, and Catch It When It Goes Wrong
## MPS311/439 Machine Learning
**Lesson 9 Lab Session | Core: about 40 minutes**

## What this lab is for

The lecture explained why a neural network works. Today you use it as a tool, on the network of Assignment 2, Section 4: train it and decide how many epochs it needs, check three versions of the code that an AI wrote, see how much the result moves with the random seed, and read the test set once. All the code is given. You run it, change the value marked `← change`, and decide what the numbers tell you. After each Part, **write down** one number from your output and one sentence.

Each Part ends with a folded **Going further** box: open questions with no single answer, for when you finish early. The folded **Try with AI** boxes are also for when you finish early; the **Advanced Study** (on the web page) is for after class.

## How to work

1. This notebook already has all the code. Save a copy first (**File → Save a copy in Drive**). Run the cells from top to bottom: later cells use names made by earlier ones. You do not need a GPU.
2. Write your answers in the *Your answer* cells.
3. **Run, and you should see** tells you what your output should show, so you know you are on track. Training uses random numbers, and Colab does the arithmetic in a slightly different order from the computer that made this page: a difference in the third decimal place is fine.

## Part 0: Set up (3 minutes)

Run these two cells without reading them closely. The first is the import cell; the second is the data block. It downloads about 11 MB the first time.

In [ ]:
import numpy as np                                        # numbers and arrays
import matplotlib.pyplot as plt                           # plots
import requests                                           # to download a file (only for the Appendix)
from tensorflow import keras                              # today's tool: neural networks

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()   # ← change: which dataset (if this fails, see the Appendix)
X = x_train.reshape(-1, 784) / 255                        # the 60,000 training images, 784 pixels each, scaled to 0 to 1
y = y_train                                               # ← change: the labels (the digit, 0 to 9)
X_fit = X[:50000]                                         # the first 50,000 images: fit the models
y_fit = y[:50000]                                         # and their digits
X_val = X[50000:]                                         # the last 10,000 images: compare and choose
y_val = y[50000:]                                         # and their digits
X_test = x_test.reshape(-1, 784) / 255                    # 10,000 test images: read once, in Part 4
print(X_fit.shape, X_val.shape, X_test.shape)             # images and pixels in each part
print(y_fit[:10])                                         # the first ten labels

**Run, and you should see:** `(50000, 784) (10000, 784) (10000, 784)`, then `[5 0 4 1 9 2 1 3 1 4]`. This is the same split as Assignment 2.

## Part 1: Train the network of Assignment 2, and decide how long (13 minutes)

Assignment 2 asks for a network with two hidden layers of 128 ReLU units and 10 softmax outputs, and for Figure 2: the training and validation curves. Before you train for 20 epochs, you want to know what the curves will tell you. Train for 8 epochs (about one minute in Colab; nothing is printed while it trains, so wait for the cell to finish):

In [ ]:
keras.utils.set_random_seed(0)                            # fix the random starting weights and batches
model = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in
    keras.layers.Dense(128, activation="relu"),           # ← change: hidden layer 1, 128 ReLU units
    keras.layers.Dense(128, activation="relu"),           # hidden layer 2, 128 ReLU units
    keras.layers.Dense(10, activation="softmax"),         # output: one probability per digit
])
model.compile(optimizer="adam",                           # gradient descent that sets its own step sizes
              loss="sparse_categorical_crossentropy",     # cross-entropy; the labels are integers 0 to 9
              metrics=["accuracy"])                       # also report accuracy
hist = model.fit(X_fit, y_fit, epochs=8,                  # ← change: the number of passes over the fitting images
                 validation_data=(X_val, y_val),          # loss and accuracy on the validation images after every epoch
                 verbose=0)                               # print nothing while training
h = hist.history                                          # four lists: loss, val_loss, accuracy, val_accuracy
print(model.count_params())                               # how many weights the network has
print(np.round(h["val_accuracy"], 4))                     # validation accuracy after each epoch

**Run, and you should see:** `118282`, then eight validation accuracies from about `0.9645` to `0.9743`; the fifth is about `0.9700`.

Now draw the two plots of Figure 2:

In [ ]:
epochs = np.arange(1, len(h["loss"]) + 1)                 # 1, 2, ..., the number of epochs
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))         # two plots side by side
axes[0].plot(epochs, h["loss"], marker="o", label="training")         # left: the training loss
axes[0].plot(epochs, h["val_loss"], marker="o", label="validation")   # and the validation loss
axes[0].set_xlabel("epoch")                               # the horizontal axis
axes[0].set_ylabel("loss")                                # the vertical axis
axes[0].legend()                                          # which line is which
axes[1].plot(epochs, h["accuracy"], marker="o", label="training")     # right: the training accuracy
axes[1].plot(epochs, h["val_accuracy"], marker="o", label="validation")   # and the validation accuracy
axes[1].set_xlabel("epoch")                               # the horizontal axis
axes[1].set_ylabel("accuracy")                            # the vertical axis
axes[1].legend()                                          # which line is which
plt.show()                                                # draw both plots
best = np.argmin(h["val_loss"]) + 1                       # the epoch with the lowest validation loss (epochs count from 1)
print(best, round(min(h["val_loss"]), 3))                 # that epoch and its validation loss

**Run, and you should see:** two plots, then `4 0.093`.

**Write down:** the epoch with the lowest validation loss, and the validation accuracy at that epoch. In one sentence: after that epoch, is the network learning more about digits, or memorising the fitting images? What in the plots is your evidence?

*Your answer:* 

In Assignment 2 you train the 20 epochs it asks for and describe both curves; here you practise how to choose the number of epochs yourself, which Part 4 uses.

**Think about** (with your neighbour, no writing): Assignment 2 asks for 20 epochs. What will Figure 2 show after epoch 8, and what should your two or three sentences about it say?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** the weights of the first 10 units of the first hidden layer, each drawn as a 28 × 28 image.
> 2. **Tell the AI:** `model.layers[0].get_weights()[0]` is a 784 × 128 array: one column of 784 weights per unit.
> 3. **Ask:** "Write simple Python, one step per line, that reshapes each of the first 10 columns to 28 × 28 and shows them in one row with `cmap='RdBu_r'`, with 0 in the middle of the colour scale."
> 4. **Check:** 10 red and blue patterns. None looks like a whole digit; most respond to a few areas of the image.

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** which digits the network confuses: a 10 × 10 confusion matrix on the validation images.
> 2. **Tell the AI:** `model` is the trained network; `model.predict(X_val)` gives 10 probabilities per image; `y_val` holds the true digits.
> 3. **Ask:** "Write simple Python that takes the most probable digit of each row of `model.predict(X_val)`, prints `sklearn.metrics.confusion_matrix(y_val, predicted)`, and prints the three largest numbers off the diagonal with their true and predicted digits."
> 4. **Check:** the diagonal adds up to `9743`. The largest mistakes are a 3 read as a 5 (18 images) and a 4 read as a 9 (17).

> **Going further (optional)**
>
> - Change the first hidden layer to 64 or 256 units and run both cells of Part 1 again. Does the epoch with the lowest validation loss move? Is the change in validation accuracy larger than what a seed does (Part 3)?
> - Let the training stop by itself when the validation loss stops improving:
> 
>

In [ ]:
stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)   # stop after 3 epochs without a lower validation loss; keep the best weights
keras.utils.set_random_seed(0)                            # the same start as Part 1
model = keras.Sequential([
    keras.layers.Input((784,)),                           # the network of Part 1
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 1
hist = model.fit(X_fit, y_fit, epochs=20, validation_data=(X_val, y_val), verbose=0, callbacks=[stop])   # at most 20 epochs
print(len(hist.history["loss"]), model.evaluate(X_val, y_val, verbose=0))   # epochs it ran; validation loss and accuracy

>   At which epoch did it stop, and which epoch's weights did it keep?


## Part 2: Check the code an AI wrote (15 minutes)

You asked an AI for "the network of Assignment 2, trained for 5 epochs, scored on the validation set". It gave three versions. Each changes one line of Part 1, and each runs without an error. Run each one (about half a minute each) and compare it with epoch 5 of Part 1: validation accuracy `0.9700`, training accuracy `0.9871`.

**Version A**

In [ ]:
pixels = x_train.reshape(-1, 784)                         # the 784 pixels of each training image
keras.utils.set_random_seed(0)                            # the same start as Part 1
model_a = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in
    keras.layers.Dense(128, activation="relu"),           # hidden layer 1
    keras.layers.Dense(128, activation="relu"),           # hidden layer 2
    keras.layers.Dense(10, activation="softmax"),         # output
])
model_a.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 1
hist_a = model_a.fit(pixels[:50000], y_fit, epochs=5,     # the first 50,000 images
                     validation_data=(pixels[50000:], y_val), verbose=0)   # the last 10,000
print(round(hist_a.history["loss"][0], 2), round(hist_a.history["accuracy"][-1], 4), round(hist_a.history["val_accuracy"][-1], 4))   # first training loss; last training and validation accuracy

**Version B**

In [ ]:
keras.utils.set_random_seed(0)                            # the same start as Part 1
model_b = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in
    keras.layers.Dense(128),                              # hidden layer 1
    keras.layers.Dense(128),                              # hidden layer 2
    keras.layers.Dense(10, activation="softmax"),         # output
])
model_b.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 1
hist_b = model_b.fit(X_fit, y_fit, epochs=5, validation_data=(X_val, y_val), verbose=0)   # 5 epochs
print(round(hist_b.history["loss"][0], 2), round(hist_b.history["accuracy"][-1], 4), round(hist_b.history["val_accuracy"][-1], 4))   # first training loss; last training and validation accuracy

**Version C**

In [ ]:
keras.utils.set_random_seed(0)                            # the same start as Part 1
model_c = keras.Sequential([
    keras.layers.Input((784,)),                           # 784 pixels in
    keras.layers.Dense(128, activation="relu"),           # hidden layer 1
    keras.layers.Dense(128, activation="relu"),           # hidden layer 2
    keras.layers.Dense(10, activation="softmax"),         # output
])
model_c.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 1
hist_c = model_c.fit(X_fit, y_fit, epochs=5, validation_data=(X_fit, y_fit), verbose=0)   # 5 epochs
print(round(hist_c.history["loss"][0], 2), round(hist_c.history["accuracy"][-1], 4), round(hist_c.history["val_accuracy"][-1], 4))   # first training loss; last training and validation accuracy

**Write down:** one line per version: its validation accuracy, then *wrong* or *allowed but worse*, and a few words why. Do not judge by the score alone. For each version find the line that differs from Part 1 and ask two questions: **what numbers does the network see, and which images is "validation" computed on?** One version scores *higher* than Part 1: that is the real result, not a copying mistake. *Wrong* means the network or the data is not what was asked for (a different model, or validation on the wrong images); *allowed but worse* means the right network on the right images, only prepared less well.

*Your answer:* 

**Think about** (with your neighbour, no writing): Version C scored higher than your own code. If someone only showed you their Figure 2, what would make you suspicious?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** to know why Version B runs without an error or a warning.
> 2. **Tell the AI:** the line is `keras.layers.Dense(128)`.
> 3. **Ask:** "What activation does `keras.layers.Dense` use when `activation` is not given? Show me the line in the Keras documentation."
> 4. **Check:** the default is `None`: no activation, each unit passes on its weighted sum. Compare with Section 3 of the Note: a line on top of lines is still a line.

> **Going further (optional)**
>
> - Write your own version D that differs from Part 1 in one line, runs without an error and is wrong (for example, something to do with `X_test` or `y_val`). Give it to your neighbour. Can they find the line from the code alone? From the score alone?
> - In Version A, change `pixels[:50000]` to `pixels[:50000] / 255` and the same for the validation images. Does the score come back to Part 1's?


## Part 3: Is a lead of 0.1 percentage points real? (6 minutes)

A classmate changed the network and scores 0.1 percentage points above you. Before you believe them, find out how much the result moves when **nothing** changes but the random seed. The seed fixes the random starting weights and which 32 images go into each step. Train the network of Part 1 for 5 epochs with two other seeds (about one minute). After 5 epochs the networks are still moving, so the spread is larger than the 0.26 points the lecture showed after 20 epochs:

In [ ]:
for seed in [1, 2]:                                       # ← change: the seeds to try (seed 0 is Part 1)
    keras.utils.set_random_seed(seed)                     # only the seed changes
    other = keras.Sequential([
        keras.layers.Input((784,)),                       # the network of Part 1
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    other.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 1
    hist_other = other.fit(X_fit, y_fit, epochs=5, validation_data=(X_val, y_val), verbose=0)       # 5 epochs
    print(seed, round(hist_other.history["val_accuracy"][-1], 4))   # the seed and its validation accuracy after 5 epochs

**Run, and you should see:** `1 0.976` and `2 0.9724` (about).

**Write down:** the largest difference between seeds 0 (Part 1: 0.9700), 1 and 2, in percentage points. In one sentence: is your classmate's 0.1-point lead evidence that their network is better?

*Your answer:* 

**Think about** (with your neighbour, no writing): Assignment 2 uses `set_random_seed(0)`. Does fixing the seed make the result more *correct*, or only repeatable?

> **Going further (optional)**
>
> - Add seeds 3 and 4. Does the spread grow, shrink or stay about the same?
> - How large must a lead be before you believe it from one seed each? How would you decide with five seeds each?

## Part 4: The test set, once (3 minutes)

Use the number of epochs you chose in Part 1, train once more, and read the test set once.

In [ ]:
keras.utils.set_random_seed(0)                            # the same start as Part 1
final = keras.Sequential([
    keras.layers.Input((784,)),                           # the network of Part 1
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
final.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])   # as in Part 1
final.fit(X_fit, y_fit, epochs=4, verbose=0)              # ← change: the number of epochs you chose in Part 1
print(final.evaluate(X_test, y_test, verbose=0))          # loss and accuracy on the 10,000 test images, read once

**Run, and you should see:** a test accuracy of about `0.9764` with 4 epochs. Do not change the number of epochs after seeing this number: that would be choosing on the test set.

**Write down:** the test accuracy. In one sentence: why do you read it only once, after choosing?

*Your answer:* 

> **Going further (optional)**
>
> The test accuracy (0.9764) is a little higher than the validation accuracy at epoch 4 (0.9743). Does that mean 4 epochs was a better choice than you thought? Compare the gap with the seed's spread from Part 3.

# After class

## How today's lab helps with Assignment 2

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 0: the data block, first 50,000 to fit, last 10,000 to validate | Data: "A fixed split, used throughout". The assignment calls the arrays `X_train`, `y_train` (our `X_fit`, `y_fit`) and `X_val`, `y_val`; its Option 1 loads the same Keras file |
| Part 1: `Sequential`, `Dense(128, activation="relu")` twice, `Dense(10, activation="softmax")`, `compile`, `fit(validation_data=...)`, the two plots | Section 4 and **Figure 2**. Change `epochs=8` to `epochs=20` (two to three minutes in Colab) and describe both curves |
| Part 2: the three versions | Sections 1 and 4: none of these mistakes may appear in your code; divide by 255; `validation_data` is the validation set |
| Part 3: two more seeds | Randomness: "write `keras.utils.set_random_seed(0)` before you build the model"; Going further: "Train again with three different random seeds" |
| Part 4: the test set, once | Section 4: the final model is checked on `test` once, after you have chosen it on the validation set |
| Try with AI: the confusion matrix | Section 4: the most frequent kinds of mistake (in the assignment, on `test`, once) |

## Reflection

One or two sentences each.

1. Why do we decide the number of epochs from the validation curve and not from the training curve?
2. Version C of Part 2 scored higher than the correct code. Why is it still wrong?
3. How large a difference between two networks would you trust, and why?

# Appendix: only if MNIST did not download

Use this only if the data block of Part 0 stopped with a download error. Each cell below gives exactly the same images; run one, then run the data block again from its second line (`X = x_train.reshape(-1, 784) / 255`).

**Backup 1: the file on the course website** (the one Assignment 2 links to).

In [ ]:
BACKUP_URL = "https://ml.wxing.me/assignments/assignment-two/mnist.npz"
reply = requests.get(BACKUP_URL)                          # download the same file as Keras uses, from a different server
open("mnist.npz", "wb").write(reply.content)              # save it in this notebook
full = np.load("mnist.npz")                               # four arrays: x_train, y_train, x_test, y_test
x_train = full["x_train"]                                 # 60,000 training images, 28 x 28
y_train = full["y_train"]                                 # their digits
x_test = full["x_test"]                                   # 10,000 test images
y_test = full["y_test"]                                   # their digits
print(x_train.shape, x_test.shape)                        # images and pixels

**Backup 2: OpenML** (the first download in Assignment 2). It takes a minute or two.

In [ ]:
from sklearn.datasets import fetch_openml                 # downloads data sets from openml.org
pixels, labels = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True, parser="liac-arff")   # all 70,000 images, in the standard order
x_train = pixels[:60000].reshape(-1, 28, 28)              # the first 60,000 are the training images
y_train = labels[:60000].astype(int)                      # the labels arrive as text; turn them into integers
x_test = pixels[60000:].reshape(-1, 28, 28)               # the last 10,000 are the test images
y_test = labels[60000:].astype(int)                       # their digits
print(x_train.shape, x_test.shape)                        # images and pixels

**Run, and you should see:** `(60000, 28, 28) (10000, 28, 28)` from either cell.